In [2]:
import os
import sys
from pathlib import Path
from typing import List, Set

import duckdb
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)
try:
    current_path = Path.cwd()
    if current_path.name == "notebook" and current_path.parent.name == "research":
        project_root = current_path.parent.parent
    else:
        project_root = current_path

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f" Project root terdeteksi: {project_root}")

except Exception as e:
    print(f" Gagal mendeteksi root. periksa struktur directory/folder/. Error: {e}")

 Project root terdeteksi: /home/bumip/orca


In [3]:
con = duckdb.connect(database=":memory:")
RAW_LAKE_PATH = project_root / "data" / "raw"
SILVER_LAKE_PATH = project_root / "data" / "silver"
print(f"Raw Lake Path   : {RAW_LAKE_PATH}")
print(f"Silver Lake Path: {SILVER_LAKE_PATH}")

con = duckdb.connect(database=":memory:")

# Tentukan path ke Silver Lake (pastikan PROJECT_ROOT sudah didefinisikan)
RAW_LAKE_PATH = project_root / "data" / "raw"
SILVER_LAKE_PATH = project_root / "data" / "silver"
silver_glob = str(SILVER_LAKE_PATH / "**" / "*.parquet")

print("Radar DuckDB siap.")
print(f"Raw Lake Path   : {RAW_LAKE_PATH}")
print(f"Glob path: {silver_glob}")

Raw Lake Path   : /home/bumip/orca/data/raw
Silver Lake Path: /home/bumip/orca/data/silver
Radar DuckDB siap.
Raw Lake Path   : /home/bumip/orca/data/raw
Glob path: /home/bumip/orca/data/silver/**/*.parquet


In [19]:
# LANGKAH 2: Hitung Total Populasi
# Gunakan DuckDB untuk menghitung total baris dari seluruh file Parquet di Silver Lake.
query = f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
"""
result = con.execute(query).fetchone()
total_rows = result[0] if result else 0

print(f"Total baris di Silver Lake: {total_rows:,}")

Total baris di Silver Lake: 1,677,669


In [23]:
# Ambil sampel 1000 baris pertama dari seluruh data Silver Lake
# Gunakan union_by_name=true untuk menangani perbedaan skema antar file
sample_query = f"""
    SELECT * FROM read_parquet('{silver_glob}', 
                                hive_partitioning=true, 
                                union_by_name=true)
    LIMIT 1000
"""
df_sample = con.execute(sample_query).df()

# Konversi DataFrame sampel ke format CSV (string) tanpa menyimpan ke disk
csv_string = df_sample.to_csv(index=False)

# Ukur ukuran dalam byte (termasuk overhead objek Python)
sample_size_bytes = sys.getsizeof(csv_string)

# Ekstrapolasi ke total baris (total_rows dari LANGKAH 2)
estimated_total_bytes = (sample_size_bytes / 1000) * total_rows

# Konversi ke Megabyte dan Gigabyte
estimated_mb = estimated_total_bytes / (1024**2)
estimated_gb = estimated_total_bytes / (1024**3)

print(f"Ukuran sampel 1000 baris: {sample_size_bytes} bytes")
print(f"Estimasi ukuran total CSV: {estimated_mb:.2f} MB ({estimated_gb:.2f} GB)")

Ukuran sampel 1000 baris: 598923 bytes
Estimasi ukuran total CSV: 958.25 MB (0.94 GB)


In [24]:
# LANGKAH 4: Interogasi Skema Mutlak (Data Types)
# Gunakan DESCRIBE pada query yang sama dengan union_by_name untuk mendapatkan skema gabungan
describe_query = f"""
    DESCRIBE (
        SELECT * FROM read_parquet('{silver_glob}', 
                                    hive_partitioning=true, 
                                    union_by_name=true)
    )
"""
schema_df = con.execute(describe_query).df()

print("Skema gabungan seluruh file Parquet di Silver Lake:")
print("----------------------------------------------------")
for _, row in schema_df.iterrows():
    col_name = row["column_name"]
    col_type = row["column_type"]
    # Tandai tipe yang diharapkan
    if col_name == "timestamp":
        marker = " ✓ TIMESTAMP"
    elif col_type.upper() in ("DOUBLE", "FLOAT", "DECIMAL"):
        marker = " ✓ DOUBLE"
    else:
        marker = ""
    print(f"{col_name:40} : {col_type}{marker}")

Skema gabungan seluruh file Parquet di Silver Lake:
----------------------------------------------------
timestamp                                : TIMESTAMP ✓ TIMESTAMP
close_BTC                                : DOUBLE ✓ DOUBLE
close_DOGE                               : DOUBLE ✓ DOUBLE
log_BTC                                  : DOUBLE ✓ DOUBLE
log_DOGE                                 : DOUBLE ✓ DOUBLE
ret_BTC                                  : DOUBLE ✓ DOUBLE
ret_DOGE                                 : DOUBLE ✓ DOUBLE
vol_BTC_1h                               : DOUBLE ✓ DOUBLE
vol_DOGE_1h                              : DOUBLE ✓ DOUBLE
corr_DOGE_BTC_1h                         : DOUBLE ✓ DOUBLE
vol_BTC_4h                               : DOUBLE ✓ DOUBLE
vol_DOGE_4h                              : DOUBLE ✓ DOUBLE
corr_DOGE_BTC_4h                         : DOUBLE ✓ DOUBLE
vol_BTC_24h                              : DOUBLE ✓ DOUBLE
vol_DOGE_24h                             : DOUBLE ✓ DOUBLE
corr

In [25]:
# LANGKAH 5: Operasi Pembongkaran Parquet Metadata (Compression Ratio)
from pathlib import Path

metadata_query = f"""
    SELECT 
        file_name,
        row_group_id,
        row_group_num_rows,
        total_compressed_size,
        total_uncompressed_size
    FROM parquet_metadata('{silver_glob}')
    LIMIT 5
"""
meta_df = con.execute(metadata_query).df()

# Konversi ke MB
meta_df["compressed_mb"] = meta_df["total_compressed_size"] / (1024**2)
meta_df["uncompressed_mb"] = meta_df["total_uncompressed_size"] / (1024**2)
meta_df["compression_ratio"] = meta_df["uncompressed_mb"] / meta_df["compressed_mb"]

# Bersihkan nama file agar hanya nama saja
meta_df["file_name"] = meta_df["file_name"].apply(lambda x: Path(x).name)

print("Metadata Parquet (5 row group pertama):")
print("---------------------------------------")
print(
    meta_df[
        [
            "file_name",
            "row_group_id",
            "row_group_num_rows",
            "compressed_mb",
            "uncompressed_mb",
            "compression_ratio",
        ]
    ].to_string(index=False)
)

Metadata Parquet (5 row group pertama):
---------------------------------------
       file_name  row_group_id  row_group_num_rows  compressed_mb  uncompressed_mb  compression_ratio
00000000.parquet             0               44640       0.116332         0.340629           2.928072
00000000.parquet             0               44640       0.132509         0.340629           2.570603
00000000.parquet             0               44640       0.073146         0.340629           4.656840
00000000.parquet             0               44640       0.276231         0.340629           1.233130
00000000.parquet             0               44640       0.074571         0.340629           4.567864


In [26]:
# LANGKAH 6: Metadata Row dan Column per File
# Menggunakan parquet_metadata untuk aggregasi per file: total baris, jumlah kolom, ukuran, rasio kompresi

file_meta_query = f"""
    SELECT 
        file_name,
        COUNT(DISTINCT row_group_id) AS row_groups,
        SUM(row_group_num_rows) AS total_rows,
        COUNT(DISTINCT column_id) AS num_columns,
        SUM(total_compressed_size) / (1024*1024) AS compressed_mb,
        SUM(total_uncompressed_size) / (1024*1024) AS uncompressed_mb,
        SUM(total_uncompressed_size) / NULLIF(SUM(total_compressed_size), 0) AS compression_ratio
    FROM parquet_metadata('{silver_glob}')
    GROUP BY file_name
    ORDER BY file_name
    LIMIT 10
"""
file_meta_df = con.execute(file_meta_query).df()

# Bersihkan nama file
file_meta_df["file_name"] = file_meta_df["file_name"].apply(lambda x: Path(x).name)

print("Metadata per File (10 file pertama):")
print("-------------------------------------")
print(file_meta_df.to_string(index=False))

Metadata per File (10 file pertama):
-------------------------------------
       file_name  row_groups  total_rows  num_columns  compressed_mb  uncompressed_mb  compression_ratio
00000000.parquet           1    937440.0           21       4.939034         6.472068           1.310391
00000000.parquet           1    846720.0           21       4.453983         5.845847           1.312499
00000000.parquet           1    935760.0           21       4.901885         6.460471           1.317956
00000000.parquet           1    907200.0           21       4.778171         6.263328           1.310821
00000000.parquet           1    937440.0           21       4.768294         6.472068           1.357313
00000000.parquet           1    907200.0           21       4.643754         6.263328           1.348764
00000000.parquet           1    937440.0           21       4.784846         6.472068           1.352618
00000000.parquet           1    937440.0           21       4.687120         6.472068

In [7]:
parquet_files = sorted(SILVER_LAKE_PATH.glob("**/*.parquet"))
print(f"📄 Ditemukan {len(parquet_files)} file .parquet")

📄 Ditemukan 39 file .parquet


In [8]:
def extract_symbols_from_schema(file_path: Path) -> Set[str]:
    """
    Baca schema parquet dan ekstrak simbol dari kolom spread_* atau z_score_*.

    Args:
        file_path: Path ke file parquet.

    Returns:
        Set berisi simbol-simbol unik yang ditemukan dalam file tersebut.
    """
    try:
        schema = pq.read_schema(file_path)
    except Exception as e:
        raise RuntimeError(f"Gagal membaca schema {file_path}: {e}")

    symbols = set()
    for field in schema.names:
        # Cari kolom dengan pola spread_XXXX atau z_score_XXXX
        if field.startswith("spread_"):
            symbol = field.replace("spread_", "")
            symbols.add(symbol)
        elif field.startswith("z_score_"):
            symbol = field.replace("z_score_", "")
            symbols.add(symbol)
    return symbols

In [9]:
all_symbols: Set[str] = set()
file_count = 0
error_files = []

for file in parquet_files:
    try:
        symbols_in_file = extract_symbols_from_schema(file)
        all_symbols.update(symbols_in_file)
        file_count += 1
    except Exception as e:
        error_files.append((str(file), str(e)))
        print(f"⚠️  Gagal memproses {file.name}: {e}")
        continue

print(f"\n✅ Berhasil memproses {file_count} dari {len(parquet_files)} file.")
if error_files:
    print(f"⚠️  {len(error_files)} file gagal diproses.")


✅ Berhasil memproses 39 dari 39 file.


In [12]:
symbol_list = sorted(list(all_symbols))
print(f"\n🔢 Jumlah simbol unik ditemukan: {len(symbol_list)}")
print("📋 Daftar simbol:")
for i, sym in enumerate(symbol_list, 1):
    print(f"   {i:3d}. {sym}")

AVAILABLE_SYMBOLS = symbol_list
print(f"\n💾 Daftar simbol tersedia di variabel `AVAILABLE_SYMBOLS`.")


🔢 Jumlah simbol unik ditemukan: 14
📋 Daftar simbol:
     1. ARB_USDT
     2. DOGE
     3. DOGE_USDT
     4. ETH_USDT
     5. FET_USDT
     6. FLOKI_USDT
     7. LDO_USDT
     8. OP_USDT
     9. PEPE_USDT
    10. RENDER_USDT
    11. SHIB_USDT
    12. SOL_USDT
    13. TAO_USDT
    14. WIF_USDT

💾 Daftar simbol tersedia di variabel `AVAILABLE_SYMBOLS`.
